In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .appName("SimpleExample") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 10:19:27 WARN Utils: Your hostname, oden-Valhalla, resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlp3s0)
26/03/07 10:19:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 10:19:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
df_from_file = spark.read.parquet("./data/dim_supplier" )
print(df_from_file.count())


30000


In [15]:
df_part = spark.read.parquet("./data/dim_part" )
print(df_part.count())

4000000


In [ ]:
# Load tables needed for the query
df_fact_sales = spark.read.parquet("./data/fact_sales")
df_customer = spark.read.parquet("./data/dim_customer")

# Reuse existing supplier dataframe if available
supplier_df = df_supplier if "df_supplier" in globals() else df_from_file

# Register temp views
df_fact_sales.createOrReplaceTempView("fact_sales")
df_customer.createOrReplaceTempView("dim_customer")
supplier_df.createOrReplaceTempView("dim_supplier")

# Execute query
result_df = spark.sql("""
SELECT 
    c.nation_name, 
    s.supplier_name,
    SUM(f.quantity) AS sum_qty,
    SUM(f.extended_price) AS sum_base_price,
    SUM(f.extended_price * (1 - f.discount)) AS sum_disc_price,
    SUM(f.extended_price * (1 - f.discount) * (1 + f.tax)) AS sum_charge,
    AVG(f.quantity) AS avg_qty,
    AVG(f.extended_price) AS avg_price,
    AVG(f.discount) AS avg_disc,
    COUNT(*) AS count_order
FROM fact_sales f
JOIN dim_customer c ON f.cust_key = c.cust_key
JOIN dim_supplier s ON f.supp_key = s.supp_key
WHERE f.ship_date <= date_sub(to_date('1998-12-01'), 90)
GROUP BY c.nation_name, s.supplier_name
ORDER BY c.nation_name, s.supplier_name
""")

result_df.show(truncate=False)